# LangChain: Memory

## Outline
* ConversationBufferMemory
* ConversationBufferWindowMemory
* ConversationTokenBufferMemory
* ConversationSummaryMemory

LangChain 的 Memory（记忆）模块负责让对话链在多轮对话之间"记住"之前说过的内容。
不同的 Memory 实现在"记多少、怎么压缩"上策略不同：全量保存 / 只保留最近 k 轮 / 按 token 数截断 / 用 LLM 做摘要压缩。

> 注：`ConversationChain`、`ConversationBufferMemory` 等在当前 langchain 版本中已被标记为 deprecated
> （官方建议改用 `langchain.agents.create_agent` + checkpointer），但目前仍可正常运行，只是会有 DeprecationWarning，
> 本 notebook 遵循原课程结构保留这些 API 并添加中文注释说明。

In [ ]:
# 加载 .env 中的环境变量（如 OPENAI_API_KEY）
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

# 屏蔽掉运行过程中一些不影响结果的警告信息（如 LangChainDeprecationWarning），让输出更干净
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# account for deprecation of LLM model
# 根据当前日期判断该用哪个 gpt-3.5-turbo 版本名（2024-06-12 之后旧的 0301 版本会被下线）
# 现在（2026 年）早已过了这个日期，会走 if 分支选用 "gpt-3.5-turbo"，逻辑本身没问题
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

In [ ]:
# ChatOpenAI 的正确导入路径是独立的 langchain_openai 包（同 L1 的修复）
from langchain_openai import ChatOpenAI
# 【版本兼容性修复】原代码是 from langchain_community.chains import ConversationChain，
# 实测会报 ImportError: cannot import name 'ConversationChain' from 'langchain_community.chains'。
# ConversationChain 属于"经典链"，在新版拆分中位于 langchain_classic.chains
from langchain_classic.chains import ConversationChain
# ConversationBufferMemory：把所有历史对话原样、完整地存起来，不做任何压缩
from langchain_classic.memory import ConversationBufferMemory

In [ ]:
# ConversationChain = LLM + Memory 的组合：每次调用都会自动把历史对话拼进 prompt 再发给模型
# verbose=True 会打印出每次实际发给模型的完整 prompt（含历史记录），便于观察 memory 的效果
llm = ChatOpenAI(temperature=0.0, model=llm_model)
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm,
    memory = memory,
    verbose=True
)

In [ ]:
# 第一轮对话：告诉模型自己的名字
# 【提示】.predict() 是 Chain 的旧式调用方法，在当前版本中仍可用，但已被标记为 deprecated，
# 新代码推荐使用 .invoke({"input": ...}) 并从返回的 dict 里取 "response" 字段。
# 这里保留 .predict() 是为了贴近原课程写法，实际会在发起真实网络请求时因假 key 而报认证错误。
conversation.predict(input="Hi, my name is Andrew")

In [ ]:
# 第二轮：问一个跟上下文无关的问题，此时 memory 里已经包含第一轮的对话历史
conversation.predict(input="What is 1+1?")

In [ ]:
# 第三轮：问模型"我叫什么名字"——如果 memory 生效，模型应该能从历史记录里回忆起 "Andrew"
conversation.predict(input="What is my name?")

In [ ]:
# memory.buffer 是内部保存的原始对话历史字符串（Human/AI 交替）
print(memory.buffer)

In [ ]:
# load_memory_variables({}) 是 Memory 对外的标准接口：返回一个 dict，
# key 是 prompt 模板里要用到的变量名（默认是 "history"），value 是格式化好的历史对话文本
memory.load_memory_variables({})

In [ ]:
# 重新创建一个空的 memory，接下来手动演示 save_context 的用法（不经过 ConversationChain）
memory = ConversationBufferMemory()

In [ ]:
# save_context(inputs, outputs) 手动往 memory 里追加一轮"人类输入 -> AI 输出"记录
# 这是纯本地操作，不发起网络请求，可以直接验证 memory 的存储行为
memory.save_context({"input": "Hi"},
                    {"output": "What's up"})

In [ ]:
print(memory.buffer)
memory.load_memory_variables({})

In [ ]:
# 再追加一轮对话，验证 memory 是"累加"而不是"覆盖"
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})

In [ ]:
# 此时应该能看到两轮对话都被保留
memory.load_memory_variables({})

## ConversationBufferWindowMemory

与 ConversationBufferMemory 全量保存不同，ConversationBufferWindowMemory 只保留最近 k 轮对话，
更早的历史会被自动丢弃——适合控制上下文长度、节省 token。

In [ ]:
# 同样从 langchain_classic.memory 导入（旧版路径 langchain.memory 在当前版本已不存在）
from langchain_classic.memory import ConversationBufferWindowMemory

In [ ]:
# k=1 表示只保留最近 1 轮（1 次人类输入 + 1 次 AI 输出）
memory = ConversationBufferWindowMemory(k=1)

In [ ]:
# 连续保存两轮对话，验证只有最近一轮会保留在 load_memory_variables 的结果里
memory.save_context({"input": "Hi"},
                    {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})

In [ ]:
# 预期：由于 k=1，第一轮 "Hi / What's up" 已经被丢弃，只剩下第二轮
memory.load_memory_variables({})

In [ ]:
# 把 window memory 接入 ConversationChain，实际对话时验证"只记得最近一轮"的效果
llm = ChatOpenAI(temperature=0.0, model=llm_model)
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm,
    memory = memory,
    verbose=False
)

In [ ]:
conversation.predict(input="Hi, my name is Andrew")

In [ ]:
conversation.predict(input="What is 1+1?")

In [ ]:
# 因为 k=1，此时 memory 里已经不包含第一轮"我叫 Andrew"的记录，模型大概率答不出名字
conversation.predict(input="What is my name?")

## ConversationTokenBufferMemory

按 token 数量而不是"轮数"来限制记忆长度：超过 max_token_limit 时，从最早的历史开始丢弃，
直到剩余历史的 token 数不超过限制。这个过程是纯本地计算（用 tiktoken 数 token），不需要联网。

In [ ]:
# ConversationTokenBufferMemory 从 langchain_classic.memory 导入
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_openai import OpenAI
llm = ChatOpenAI(temperature=0.0, model=llm_model)

In [ ]:
# max_token_limit=50：历史记录超过 50 个 token 后，最早的对话会被截断丢弃
# 这里已用本地 venv 实际验证：save_context 全程无需网络请求（token 计数走本地 tiktoken），
# 最终 load_memory_variables 只保留了最后一两轮，符合"按 token 数截断"的预期
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=50)
memory.save_context({"input": "AI is what?!"},
                    {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"},
                    {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"},
                    {"output": "Charming!"})

In [ ]:
# 最早的一两轮对话应该已经因为超出 token 限制被丢弃
memory.load_memory_variables({})

## ConversationSummaryMemory

前面几种 Memory 都是"截断"策略（丢弃旧记录），而 ConversationSummaryBufferMemory 更聪明：
当历史超出 token 限制时，会调用 LLM 把最早的那部分对话"压缩成一句摘要"，而不是直接丢弃，
从而在节省 token 的同时尽量保留信息。因为要调用 LLM 生成摘要，这一部分需要真实网络请求。

In [ ]:
# 【版本兼容性修复】同前，正确路径是 langchain_classic.memory
from langchain_classic.memory import ConversationSummaryBufferMemory

In [ ]:
# create a long string
# 构造一段较长的日程安排文本，用来触发"超出 token 限制、需要生成摘要"的场景
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

# max_token_limit=100：一旦历史超过 100 个 token，最早的部分会被 LLM 总结成摘要而不是直接丢弃
# 【注意】save_context 触发摘要生成需要调用 LLM，这里会因假 API Key 在联网阶段报错，属预期行为
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
memory.save_context({"input": "What is on the schedule today?"},
                    {"output": f"{schedule}"})

In [ ]:
# 预期效果：早期的闲聊被压缩成一句摘要，加上最近一轮的完整日程内容
memory.load_memory_variables({})

In [ ]:
# 把摘要型 memory 接入 ConversationChain，实际对话验证效果
conversation = ConversationChain(
    llm=llm,
    memory = memory,
    verbose=True
)

In [ ]:
# 问一个需要结合"日程安排"上下文才能回答好的问题，验证摘要记忆是否保留了关键信息
conversation.predict(input="What would be a good demo to show?")

In [ ]:
# 再次查看当前 memory 内容，验证最新一轮对话也被写入了摘要/缓冲区
memory.load_memory_variables({})